## Sync Delta Tables to Lakebase

Creates synced tables in Lakebase PostgreSQL from Delta source tables,
and a native `cat_reviews` table for writable human reviews.

Synced tables (read-only): invoices, categories, cat_bootstrap, cat_catboost, cat_vectorsearch
Native table (writable): cat_reviews

Tested on Serverless v4.

In [0]:
%pip install pyyaml "databricks-sdk>=0.50.0" psycopg2-binary
%restart_python

In [0]:
from src.config import load_config

config = load_config()
catalog = config.catalog
schema = config.schema_name
instance_name = config.lakebase_instance
dbname = config.lakebase_dbname

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Lakebase instance: {instance_name}")
print(f"Lakebase DB: {dbname}")

### Create synced tables from Delta sources

Each synced table mirrors a Delta table into Lakebase PostgreSQL as a read-only table.
Uses `SNAPSHOT` mode -- trigger a re-sync after each pipeline run.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import (
    SyncedDatabaseTable,
    SyncedTableSpec,
    NewPipelineSpec,
    SyncedTableSchedulingPolicy,
)

w = WorkspaceClient()

TABLES_TO_SYNC = [
    {"source": "invoices", "pk": ["order_id"]},
    {"source": "categories", "pk": ["category_level_1", "category_level_2", "category_level_3"]},
    {"source": "cat_bootstrap", "pk": ["order_id"]},
    {"source": "cat_catboost", "pk": ["order_id"]},
    {"source": "cat_vectorsearch", "pk": ["order_id"]},
]

for table_info in TABLES_TO_SYNC:
    source_name = table_info["source"]
    pk_cols = table_info["pk"]
    source_full = f"{catalog}.{schema}.{source_name}"
    synced_full = f"{catalog}.{schema}.{source_name}_synced"

    print(f"Syncing {source_full} -> {synced_full} (PK: {pk_cols})")

    try:
        existing = w.database.get_synced_database_table(name=synced_full)
        print(f"  Already exists: {existing.data_synchronization_status.detailed_state}")
        continue
    except Exception:
        pass

    try:
        w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=synced_full,
                database_instance_name=instance_name,
                logical_database_name=dbname,
                spec=SyncedTableSpec(
                    source_table_full_name=source_full,
                    primary_key_columns=pk_cols,
                    scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
                    create_database_objects_if_missing=True,
                    new_pipeline_spec=NewPipelineSpec(
                        storage_catalog=catalog,
                        storage_schema=schema,
                    ),
                ),
            )
        )
        print(f"  Created synced table: {synced_full}")
    except Exception as e:
        print(f"  Error creating {synced_full}: {e}")

### Create native cat_reviews table in Lakebase

The reviews table is writable (not synced) so the app can INSERT reviews directly.

In [0]:
import uuid
import psycopg2

db_instance = w.database.get_database_instance(name=instance_name)
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[instance_name],
)
user = w.current_user.me()
user_email = user.emails[0].value if user.emails else user.user_name

conn = psycopg2.connect(
    host=db_instance.read_write_dns,
    dbname=dbname,
    user=user_email,
    password=cred.token,
    sslmode="require",
)

with conn.cursor() as cur:
    cur.execute(f"""
        CREATE SCHEMA IF NOT EXISTS \"{schema}\"
    """)
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS \"{schema}\".\"cat_reviews\" (
            review_id TEXT,
            order_id TEXT,
            source TEXT,
            reviewer TEXT,
            review_date DATE,
            original_level_1 TEXT,
            original_level_2 TEXT,
            original_level_3 TEXT,
            reviewed_level_1 TEXT,
            reviewed_level_2 TEXT,
            reviewed_level_3 TEXT,
            review_status TEXT,
            comments TEXT,
            created_at TIMESTAMPTZ
        )
    """)
    conn.commit()

conn.close()
print(f"Created native cat_reviews table in {instance_name}/{dbname}/{schema}")

### Verify synced tables

Check sync status for all created tables.

In [0]:
for table_info in TABLES_TO_SYNC:
    synced_full = f"{catalog}.{schema}.{table_info['source']}_synced"
    try:
        status = w.database.get_synced_database_table(name=synced_full)
        state = status.data_synchronization_status.detailed_state
        msg = status.data_synchronization_status.message or ""
        print(f"{synced_full}: {state} - {msg}")
    except Exception as e:
        print(f"{synced_full}: NOT FOUND - {e}")